# Real-data example: Iris morphology across species

This notebook shows how PyStars can be used on a small, real biological dataset.
We compare four continuous flower morphology measurements across three *Iris* species.

**Research question:** Do sepal and petal measurements differ between *Iris setosa*, *Iris versicolor*, and *Iris virginica*?

**Data source:** Fisher, R. A. (1936). *Iris* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C56C76

The dataset is licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/legalcode).

In [1]:
import pandas as pd

import pystars as ps

## Load the data

The UCI Iris CSV has no header, so we supply column names manually.

In [2]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data"
column_names = ["sepal_length", "sepal_width", "petal_length", "petal_width", "species"]

df = pd.read_csv(url, header=None, names=column_names)

df.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


In [3]:
# Sample sizes per species
df["species"].value_counts()

species
Iris-setosa        50
Iris-versicolor    50
Iris-virginica     50
Name: count, dtype: int64

## Primary analysis: petal length

We start with a single measurement and let PyStars choose the appropriate test based on the assumptions.

In [4]:
result = ps.test(df, value="petal_length", group="species")
result.show()

╭───────────────────────────────────────────────── Welch's ANOVA ─────────────────────────────────────────────────╮
│  Field      Value                                                                                               │
│  Test       Welch's ANOVA                                                                                       │
│  Statistic  1827                                                                                                │
│  p-value    <0.0001                                                                                             │
│  np2        0.9413                                                                                              │
│  Assumption      Method   p-value  Verdict                                                                      │
│  normality       shapiro  0.05465  not rejected                                                                 │
│  equal variance  levene   <0.0001  rejected                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [5]:
# Tidy one-row export
result.to_dataframe()

,test,statistic,p_value,np2,normality_method,normality_statistic,normality_p,equal_variance_method,equal_variance_statistic,equal_variance_p,posthoc
0,Welch's ANOVA,1826.580952,2.853131e-66,0.941319,shapiro,0.954946,0.05465,levene,19.720055,2.589296e-08,Games-Howell


In [6]:
# Pairwise post-hoc comparisons selected by the dispatcher
result.pairwise

## Batch analysis across all morphology features

For exploratory work it is convenient to run the same workflow on several measurements and collect the results in one table.

In [7]:
features = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
results = [ps.test(df, value=f, group="species") for f in features]

summary = ps.to_dataframe(results)
summary = summary.sort_values("p_value").reset_index(drop=True)
summary

,test,statistic,p_value,np2,normality_method,normality_statistic,normality_p,equal_variance_method,equal_variance_statistic,equal_variance_p,posthoc,epsilon_squared
0,Welch's ANOVA,1826.580952,2.853131e-66,0.941319,shapiro,0.954946,0.054650,levene,19.720055,2.589296e-08,Games-Howell,NaN
1,Kruskal-Wallis test,131.093353,3.415388e-29,NaN,shapiro,0.813817,0.000002,levene,19.412207,3.301950e-08,Dunn's test,0.879821
2,Welch's ANOVA,138.908285,1.505059e-28,0.618706,shapiro,0.971179,0.258315,levene,6.352720,2.258528e-03,Games-Howell,NaN
3,One-way ANOVA,47.364461,1.327917e-16,0.391881,shapiro,0.967391,0.180896,levene,0.647522,5.248270e-01,Tukey HSD,NaN


## Interpretation

All four morphology measurements differ strongly across species. The dispatcher selected Welch's ANOVA for most features because variances are unequal between species; it then ran the matching post-hoc test (Games–Howell or Dunn) when the omnibus test was significant.

**Caveat:** This is exploratory analysis. The four tests were not corrected for multiple comparisons, so the smallest p-values should be interpreted as descriptive rather than definitive evidence.